# 讓 Agent 接得住前後文

這份教材的主題是同一段對話如何延續。基礎模組不另外拆解，重點放在 `session_id` 與 `memory.turns`。

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

## 載入流程元件

這次只需要一條簡單流程，並明確建立一個對話記憶物件。`Workflow` 預設會使用 `InContextMemory`；這裡先手動宣告，是為了讓多輪對話由同一個 memory 物件累積這件事更清楚。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.memory import InContextMemory
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 先決定這段對話的識別碼與記憶

同一個使用者、同一段對話，應該使用同一個 `session_id`。同一個 memory 物件會保存這段對話的 turns；換掉 `session_id` 或 memory，就等於切到另一段上下文。

In [ ]:
session_id = 'demo-user-001'
memory = InContextMemory()

session_id

## 建立可以重複使用的流程

流程本身不用每輪重建。多輪上下文由 `memory_type` 接到的記憶策略或 memory 物件負責；這裡把剛剛建立的 `memory` 交給 `Workflow`，後面就能直接從同一個物件查看完整上下文。

In [ ]:
workflow = Workflow(
    workflow_name='多輪問答 Agent',
    memory_type=memory,
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(items=[
        {'keywords': ['專案代號', 'aurora'], 'content': '使用者提到的專案代號是 Aurora。'},
        {'keywords': ['會議', '明天'], 'content': '明天會議需要準備專案摘要。'},
    ]),
    action=DirectAnswerAction(),
)

## 第一輪：先告訴 Agent 一件事

第一輪會把使用者訊息和 Agent 回覆保存到這個 `session_id` 底下。

In [ ]:
first = workflow.run('請記住，這次專案代號是 Aurora。', session_id=session_id)
print(first.final_message)

## 第二輪：接著問後續問題

第二輪仍然使用同一個 `session_id`，所以記憶裡會保留前一輪內容。

In [ ]:
second = workflow.run('那明天會議我要準備什麼？', session_id=session_id)
print(second.final_message)

## 查看保存下來的對話紀錄

因為 `Workflow` 使用的是前面宣告的 `memory` 物件，所以兩輪執行後可以直接從 `memory` 查看完整上下文，而不是另外從 `run()` 回傳值取出記憶。

In [ ]:
print(memory.as_text_transcript())

for index, turn in enumerate(memory.turns, start=1):
    print(index, turn.role, '=>', turn.content)

## 換一段新的對話記憶看看

如果要切到另一段上下文，最清楚的方式是建立新的 memory 物件，再交給新的 `Workflow`。這可以避免不同使用者或不同任務互相污染。

In [ ]:
another_memory = InContextMemory()
another_workflow = Workflow(
    workflow_name='另一段多輪問答 Agent',
    memory_type=another_memory,
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(items=[
        {'keywords': ['專案代號', 'aurora'], 'content': '使用者提到的專案代號是 Aurora。'},
        {'keywords': ['會議', '明天'], 'content': '明天會議需要準備專案摘要。'},
    ]),
    action=DirectAnswerAction(),
)

new_session = another_workflow.run('剛剛的專案代號是什麼？', session_id='another-session')
print(new_session.final_message)
print('new memory turns:', len(another_memory.turns))
print('original memory turns:', len(memory.turns))